# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane over the others because it produces a ranked, actionable output (a review queue) rather than just a report, and it maps directly onto the kind of tabular classification/ranking work I've already been doing on Kaggle (LightGBM/XGBoost ensembles, OOF validation). Lane 1 (signal analysis) and Lane 3 (clustering) are more exploratory and end in an EDA writeup; Lane 4 (CTR scoring) is a narrower version of the same idea restricted to CTR gaps. Lane 2 lets me build the full workflow this internship teaches — baseline rule, then a model that has to beat it — while staying close to what I already know how to validate honestly (client-holdout splits, precision@K instead of accuracy). I'll confirm or change this by end of Week 4 once I've looked at the warehouse release.


In [1]:
# Quick sanity check: confirm the starter CSV loads and has the columns this lane needs
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows x Cols:", df.shape)
print("Clients:", df["client_id"].nunique())


Rows x Cols: (30000, 44)
Clients: 32


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** out of thousands of content pages, which ones should a content editor look at *first* this week?

**Unit of analysis:** one page (`content_id`) — a single content item, aggregated over its trailing 90-day window.

**Output:** a ranked review queue — top N pages, each with a priority score and a reason code (e.g. "page-one page losing impressions", "high-demand page in decline").

**Who acts, and how:** a content editor or SEO strategist at a FlyRank client. They work down the queue and decide, per page, whether to refresh, expand, protect, prune, or just monitor it. Today they either work off a simple hand-rule or gut feel, and with a 30k+ page inventory per client base, most declining pages never get looked at at all.

**Cost of a wrong call:**
- *False positive* (queue says "review this," but it wasn't actually worth it): wastes editor hours — the expensive resource here, since editor time is the bottleneck, not compute.
- *False negative* (a genuinely declining, high-value page never surfaces): a client keeps losing organic traffic on a page that could have been saved, and nobody notices until the loss is much larger.
Because editor hours are limited and finite, the practical cost of false positives is more immediate (wasted reviews), but the cost of false negatives compounds silently — so precision@K on the top of the queue matters most, and recall on high-demand pages matters second.


In [2]:
# No extra computation needed for this section - the numbers backing it come in section 3.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Numbers computed in the cell below, on the 30,000-row starter dataset (32 clients):

- **16,262 pages (54.2%)** are flagged `trend_direction == "down"` — over half the inventory is technically "declining" by the simple rule. That's too many for an editor to review by hand, which is exactly why a *ranked* queue (not just a flag) is the useful output.
- **11,156 pages (37.2%)** are declining *and* still pull real search demand (≥300 impressions in the 90-day window) — these are the pages actually worth an editor's time, as opposed to declining pages nobody was seeing anyway.
- **6,730 pages (22.4%)** are already ranking on page 1 of search *and* declining — the highest-value/highest-urgency slice, since losing a page-1 ranking is a bigger loss than losing a page-4 ranking nobody clicked on anyway.

These three numbers together are the case for the lane: the naive "declining" flag is too blunt (54% of everything), but a demand + position-aware ranking narrows that down to a manageable, genuinely prioritized queue — which is a scoring/ranking problem, not a one-line filter.


**Why ML, and not just a rule?** A plain rule already exists here — `trend_direction == "down"` — and it's *not enough on its own*: it flags 54.2% of the inventory with no ordering inside that group, so an editor still doesn't know where to start. A slightly better rule (add a demand floor, add a position check) gets to the second and third numbers above, and that's basically what the Lane 2 baseline score will be — a transparent, hand-written rule I build and evaluate honestly. Where ML (a learned ranking model) should earn its place beyond that rule is if there are more signals worth combining (freshness, word count, CTR gap, content type, intent) than I can hand-tune weights for — a case I still have to prove with a baseline-vs-model comparison in Week 4-5, not assume upfront.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_total = len(df)

# 1. How many pages are flagged as declining by the simple trend rule?
declining = df[df["trend_direction"] == "down"]
print(f"Declining (trend_direction == 'down'): {len(declining):,} of {n_total:,} rows ({len(declining)/n_total*100:.1f}%)")

# 2. Of those, how many still have real search demand (>=300 impressions/90d)?
declining_with_demand = declining[declining["impressions_90d"] >= 300]
print(f"Declining AND impressions_90d >= 300: {len(declining_with_demand):,} rows ({len(declining_with_demand)/n_total*100:.1f}%)")

# 3. Of those, how many are already on page 1 (highest urgency: losing a good ranking)?
page1_decay = df[(df["position_tier"] == "page_1") & (df["trend_direction"] == "down")]
print(f"Page-1 pages that are declining ('page-one decay risk'): {len(page1_decay):,} rows ({len(page1_decay)/n_total*100:.1f}%)")


Declining (trend_direction == 'down'): 16,262 of 30,000 rows (54.2%)
Declining AND impressions_90d >= 300: 11,156 rows (37.2%)
Page-1 pages that are declining ('page-one decay risk'): 6,730 rows (22.4%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- *Observed*: which pages, in this 90-day window, showed declining impressions alongside real search demand or a strong ranking position. These are measurements, not predictions.
- *Directional*: that pages matching certain patterns (page-1 + declining, high-demand + declining) are, on average, more worth an editor's attention than the rest of the inventory — a ranking, not a guarantee for any individual page.
- *Decision-support*: a ranked queue that helps a human prioritize where to look first. The editor still makes the call; the model doesn't decide "refresh" or "prune" on its own.

**What I will never claim:**
- That the model predicts what Google's ranking algorithm will do next, or reverse-engineers it.
- That a high score *causes* a decline, or that refreshing a page *will* fix it — this is observational data, not an experiment, so I can't make causal claims from it alone.
- That `is_declining_label` (derived from `trend_direction`, which is derived from `trend_pct`) is ground truth about page quality — it's a rule-based label about a 30/60-day impression comparison, nothing more. And per the data dictionary, `trend_direction` and `trend_pct` themselves can never be used as model features, since that would just teach the model to reproduce its own label.


In [4]:
# No extra computation needed for this section - it's a framing/claims statement, not a data check.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.